# Phase 3 Visualizations

This notebook visualizes the Team 05 Epidemic Engine end-to-end outputs: Phase 2 Spark analytics, Phase 3 outbreak predictions, vaccination change-order analytics, and Phase 2.5 / Category D data quality monitoring. It runs in demo mode by default and can query TimescaleDB when `PHASE3_LIVE_DB=1` and `DB_*` environment variables are set.

In [ ]:
import os
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import pandas as pd

USE_LIVE_DB = os.getenv('PHASE3_LIVE_DB', '0') == '1'

def live_frames():
    import psycopg2

    conn = psycopg2.connect(
        host=os.getenv('DB_HOST', 'localhost'),
        port=os.getenv('DB_PORT', '5432'),
        dbname=os.getenv('DB_NAME', 'analytics'),
        user=os.getenv('DB_USER', 'postgres'),
        password=os.getenv('DB_PASSWORD', ''),
    )
    queries = {
        'hourly_event_rate': 'SELECT * FROM hourly_event_rate ORDER BY hour;',
        'severity_trend': 'SELECT * FROM severity_trend ORDER BY window_start;',
        'bed_availability': 'SELECT * FROM bed_availability ORDER BY hour;',
        'outbreak_predictions': 'SELECT * FROM outbreak_predictions ORDER BY hour;',
        'vaccination_trend': 'SELECT * FROM vaccination_trend ORDER BY hour;',
        'data_quality_checks': 'SELECT * FROM data_quality_checks ORDER BY checked_at;',
    }
    try:
        return {name: pd.read_sql_query(sql, conn) for name, sql in queries.items()}
    finally:
        conn.close()

def demo_frames():
    base = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0) - timedelta(hours=5)
    hours = [base + timedelta(hours=i) for i in range(6)]
    regions = ['Boston', 'Cambridge', 'Worcester', 'Springfield']
    event_types = ['symptom_report', 'clinic_visit', 'hospital_admission', 'vaccination_record']

    hourly = []
    severity = []
    beds = []
    vax = []
    predictions = []
    for h_idx, hour in enumerate(hours):
        for r_idx, region in enumerate(regions):
            for e_idx, event_type in enumerate(event_types):
                hourly.append({'hour': hour, 'region': region, 'event_type': event_type, 'event_count': 4 + h_idx + r_idx + e_idx})
            severity.append({'window_start': hour, 'window_end': hour + timedelta(hours=1), 'region': region, 'event_type': 'symptom_report', 'avg_severity': min(4.0, 1.2 + 0.2*h_idx + 0.15*r_idx), 'sample_count': 8 + h_idx})
            beds.append({'hour': hour, 'region': region, 'avg_available_beds': 220 - 14*h_idx - 10*r_idx, 'min_available_beds': 180 - 18*h_idx - 12*r_idx, 'sample_count': 5 + h_idx})
            for dose in [1, 2]:
                vax.append({'hour': hour, 'region': region, 'vaccine_type': 'influenza', 'dose_number': dose, 'vaccination_count': max(0, 24 - 2*h_idx - r_idx - dose)})
            risk = min(1.0, 0.12 + 0.09*h_idx + 0.04*r_idx)
            predictions.append({'hour': hour, 'region': region, 'outbreak_risk_score': risk, 'outbreak_risk_level': 'critical' if risk >= 0.75 else 'high' if risk >= 0.5 else 'medium' if risk >= 0.25 else 'low'})

    quality = [
        {'checked_at': hours[-1], 'check_name': 'freshness', 'table_name': 'hourly_event_rate', 'status': 'PASS', 'message': 'Latest data is fresh'},
        {'checked_at': hours[-1], 'check_name': 'freshness', 'table_name': 'bed_availability', 'status': 'WARN', 'message': 'Near stale threshold'},
        {'checked_at': hours[-1], 'check_name': 'vaccination_values', 'table_name': 'vaccination_trend', 'status': 'PASS', 'message': 'Vaccination values valid'},
        {'checked_at': hours[-1], 'check_name': 'outbreak_prediction_table', 'table_name': 'outbreak_predictions', 'status': 'PASS', 'message': 'Outbreak table exists'},
    ]
    return {
        'hourly_event_rate': pd.DataFrame(hourly),
        'severity_trend': pd.DataFrame(severity),
        'bed_availability': pd.DataFrame(beds),
        'outbreak_predictions': pd.DataFrame(predictions),
        'vaccination_trend': pd.DataFrame(vax),
        'data_quality_checks': pd.DataFrame(quality),
    }

frames = live_frames() if USE_LIVE_DB else demo_frames()
print('mode:', 'live TimescaleDB' if USE_LIVE_DB else 'demo/sample data')
print('tables:', ', '.join(frames.keys()))

## 1. Hourly Event Rate by Event Type / Region

Source table: `hourly_event_rate`. This shows the volume of routed events, including the new `vaccination_record` event type after CO-2026-01.

In [ ]:
df = frames['hourly_event_rate'].copy()
pivot = df.groupby(['hour', 'event_type'])['event_count'].sum().unstack(fill_value=0)
ax = pivot.plot(figsize=(10, 4), marker='o')
ax.set_title('Hourly Event Rate by Event Type')
ax.set_xlabel('Hour')
ax.set_ylabel('Events')
plt.show()
df.groupby(['region', 'event_type'])['event_count'].sum().reset_index().head(12)

## 2. Severity Trend by Region / Event Type

Source table: `severity_trend`. Spark converts severity labels to a numeric scale and aggregates them in hourly windows.

In [ ]:
df = frames['severity_trend'].copy()
pivot = df.groupby(['window_start', 'region'])['avg_severity'].mean().unstack(fill_value=0)
ax = pivot.plot(figsize=(10, 4), marker='o')
ax.set_title('Average Severity Trend by Region')
ax.set_xlabel('Window start')
ax.set_ylabel('Average severity')
plt.show()
df.sort_values('avg_severity', ascending=False).head(10)

## 3. Bed Availability Pressure by Region

Source table: `bed_availability`. Lower `min_available_beds` indicates greater operational pressure.

In [ ]:
df = frames['bed_availability'].copy()
pivot = df.pivot_table(index='hour', columns='region', values='min_available_beds', aggfunc='min')
ax = pivot.plot(figsize=(10, 4), marker='o')
ax.set_title('Minimum Available Beds by Region')
ax.set_xlabel('Hour')
ax.set_ylabel('Minimum available beds')
plt.show()
df.sort_values('min_available_beds').head(10)

## 4. Outbreak Risk Score by Region / Hour

Source table: `outbreak_predictions`. The score combines event volume, severity, bed pressure, and vaccination activity.

In [ ]:
df = frames['outbreak_predictions'].copy()
pivot = df.pivot_table(index='hour', columns='region', values='outbreak_risk_score', aggfunc='mean')
ax = pivot.plot(figsize=(10, 4), marker='o')
ax.set_title('Outbreak Risk Score by Region')
ax.set_xlabel('Hour')
ax.set_ylabel('Risk score')
plt.show()
df.sort_values('outbreak_risk_score', ascending=False).head(10)

## 5. Vaccination Analytics by Region / Vaccine Type / Dose

Source table: `vaccination_trend`. This is the downstream output required by CO-2026-01 and uses `vaccine_type` plus `dose_number`.

In [ ]:
df = frames['vaccination_trend'].copy()
grouped = df.groupby(['region', 'vaccine_type', 'dose_number'])['vaccination_count'].sum().reset_index()
grouped['label'] = grouped['region'] + ' / ' + grouped['vaccine_type'] + ' dose ' + grouped['dose_number'].astype(str)
ax = grouped.sort_values('vaccination_count').plot(kind='barh', x='label', y='vaccination_count', figsize=(10, 5), legend=False)
ax.set_title('Vaccination Records by Region, Vaccine Type, and Dose')
ax.set_xlabel('Vaccination records')
ax.set_ylabel('')
plt.show()
grouped.sort_values('vaccination_count', ascending=False).head(10)

## 6. Pipeline / Data Quality Health Status

Source table: `data_quality_checks`. This shows whether Phase 3 can trust the analytics data it is reading.

In [ ]:
df = frames['data_quality_checks'].copy()
counts = df.groupby('status')['check_name'].count().reindex(['PASS', 'WARN', 'FAIL']).fillna(0)
ax = counts.plot(kind='bar', figsize=(6, 4), color=['#2e7d32', '#f9a825', '#c62828'])
ax.set_title('Data Quality Check Status')
ax.set_xlabel('Status')
ax.set_ylabel('Checks')
plt.show()
df.sort_values('checked_at', ascending=False).head(12)